# 10. `wait (command <N>)` (action, start)

In [1]:
from common import *

shape, log = show(10)

log shape 6: action | start | wait (command <N>)
12,689 logs (3.1%) from 256 nodes, 2003-12-26 18:27:10 to 2006-04-24 21:21:54
log 12366 at 2004-03-12 05:46:52: 257391 node-94 action start 1079070412 1 wait  (command 2616)


Wait for what? What does this mean? Need more context.

In [2]:
timeline(log.node, log.ts - day, log.ts)

,first_ts,last_ts,count,component,state,template
0,2004-03-11 14:03:54,2004-03-11 14:03:54,1,node,status,not responding
1,2004-03-11 14:04:00,2004-03-11 14:04:00,1,node,status,configured out
2,2004-03-11 14:30:53,2004-03-11 14:30:53,1,action,start,boot (command <N>)
3,2004-03-11 14:36:12,2004-03-11 14:36:12,1,action,start,wait (command <N>)
4,2004-03-11 14:36:23,2004-03-11 14:36:23,1,unix.hw,net.niff.up,NIFF: node <NODE> has detected an available network connection on network <N>.<N>.<N>.<N> via interface ee<N>
5,2004-03-11 14:36:23,2004-03-11 14:36:23,1,unix.hw,net.niff.up,NIFF: node <NODE> has detected an available network connection on network <N>.<N>.<N>.<N> via interface alt<N>
6,2004-03-11 14:37:38,2004-03-11 14:37:38,1,node,temperature,ambient=<N>
7,2004-03-11 15:34:33,2004-03-11 15:34:33,1,node,status,running
8,2004-03-11 18:47:23,2004-03-11 18:47:23,1,node,status,configured out
9,2004-03-11 18:47:34,2004-03-11 18:47:34,1,node,temperature,ambient=<N>


`wait` comes after `boot`. the pair appeared together 3 times in the past day, each time `boot` is a few minutes before `wait`.

In the previous occurences, the node status changed to `running` shortly after the `wait` command.

Btw there seems to be a sequence that appears every time after a node has rebooted, which includes something to do with `NIFF`, and temperature reporting, and then a `status: running` message.

We can see here too that every boot on this node in the past 30 days is followed by a wait, with the same command number.
- Every operator action on this node in the 30 days before:

In [3]:
actions = as_frame(logs.search(log.ts - 30 * day, log.ts, node=log.node, component="action", limit=None))
actions["verb"] = actions.message.str.split().str[0]
actions["command"] = actions.message.str.extract(r"command (\d+)")[0]
display(actions[["ts", "verb", "command"]])

,ts,verb,command
0,2004-02-11 22:34:40,boot,2152
1,2004-02-11 22:37:37,wait,2152
2,2004-02-15 17:23:34,boot,2187
3,2004-02-15 17:27:43,wait,2187
4,2004-02-21 00:25:26,boot,2236
5,2004-02-21 00:30:59,wait,2236
6,2004-02-24 18:25:46,boot,2268
7,2004-02-24 18:31:14,wait,2268
8,2004-02-25 20:12:10,boot,2296
9,2004-02-25 20:17:11,wait,2296


What are the reboots triggered by?
- Shutdown commands in the hour before:

In [4]:
as_frame(logs.search(log.ts - hour, log.ts, component="shutdown_cmd", state="new", limit=None))

,id,ts,node,component,state,message,log_shape_id
0,32428,2004-03-12 05:22:05,2591,shutdown_cmd,new,Targeting domains:node-D[0-7] and nodes:node-[0-255],19
1,32420,2004-03-12 05:22:07,2599,shutdown_cmd,new,Targeting domains:node-D7 and nodes:node-[224-255] child of command 2591,23
2,32421,2004-03-12 05:22:07,2598,shutdown_cmd,new,Targeting domains:node-D6 and nodes:node-[192-223] child of command 2591,23
3,32422,2004-03-12 05:22:07,2597,shutdown_cmd,new,Targeting domains:node-D5 and nodes:node-[160-191] child of command 2591,23
4,32423,2004-03-12 05:22:07,2596,shutdown_cmd,new,Targeting domains:node-D4 and nodes:node-[128-159] child of command 2591,23
5,32424,2004-03-12 05:22:07,2595,shutdown_cmd,new,Targeting domains:node-D3 and nodes:node-[96-127] child of command 2591,23
6,32425,2004-03-12 05:22:07,2594,shutdown_cmd,new,Targeting domains:node-D2 and nodes:node-[64-95] child of command 2591,23
7,32426,2004-03-12 05:22:07,2593,shutdown_cmd,new,Targeting domains:node-D1 and nodes:node-[32-63] child of command 2591,23
8,32427,2004-03-12 05:22:07,2592,shutdown_cmd,new,Targeting domains:node-D0 and nodes:node-[0-31] child of command 2591,23


In fact we can look at the command that triggered our wait log specifically:

In [5]:
as_frame(logs.search(log.ts - hour, log.ts, node=2616))

,id,ts,node,component,state,message,log_shape_id
0,32403,2004-03-12 05:32:49,2616,boot_cmd,new,Targeting domains:node-D2 and nodes:node-[72-95] child of command 2615,20


Our node, node-94, is in the range.

# Verdict: no action. 'wait' is the second step of a boot command: on this node every command is a boot and then a wait.
This boot follows a shutdown command for the whole cluster, so it is a planned restart.